## Voting CLF (XGB, GBC, LR) Training and Evaluation

- 1_STAGE: XGB
- 2_STAGE: Voting CLF (XGB, GBC, LR)

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# === MLflow log control ===
log_to_mlflow = True 

if log_to_mlflow:
    import mlflow
    import mlflow.sklearn
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("NBA Final Prediction 2025")

# === 1. Wczytanie danych
df = pd.read_csv("databases/nba_dataset_2010_2025.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]
df["target_binary"] = (df["target"] > 0).astype(int)

train_mask = df["season"] < 2025
test_mask = df["season"] == 2025
X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)
y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === Stage 1: klasyfikacja binarna
model_bin = XGBClassifier(
    objective="binary:logistic",
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)

sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
stage1_preds = model_bin.predict(X_test)

# === Stage 2: przygotowanie danych
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}

X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)
X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
stage1_mask = pd.Series(stage1_preds == 1, index=players_test.index)

players_stage2 = players_test[stage1_mask].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_mask].reset_index(drop=True)

# === Model ensemble
model_xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)
model_gbc = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model_logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=42
    )
)
voting_model = VotingClassifier(
    estimators=[
        ("xgb", model_xgb),
        ("gbc", model_gbc),
        ("logreg", model_logreg)
    ],
    voting="soft"
)

# === MLflow run (opcjonalnie)
if log_to_mlflow:
    with mlflow.start_run(run_name="Voting_XGB_GBC_LogReg_2025"):
        mlflow.log_params({f"stage1__{k}": v for k, v in model_bin.get_params().items()})
        mlflow.log_params({f"stage2_xgb__{k}": v for k, v in model_xgb.get_params().items()})
        mlflow.log_params({f"stage2_gbc__{k}": v for k, v in model_gbc.get_params().items()})
        mlflow.log_param("stage2_logreg_type", "StandardScaler + LogisticRegression")
        mlflow.set_tag("pipeline", "two_stage_voting_classifier")
        mlflow.set_tag("prediction_season", "2025")

        voting_model.fit(X_train_stage2, y_train_stage2)
        probas_stage2 = voting_model.predict_proba(X_test_stage2)

        df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
        df_pred["Player"] = players_stage2
        df_pred["is_rookie"] = is_rookie_stage2

        results = {
            "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
            "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
            "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
            "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
            "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
        }

        output_path = "classification_result_2025.json"
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        mlflow.log_artifact(output_path)

else:
    voting_model.fit(X_train_stage2, y_train_stage2)
    probas_stage2 = voting_model.predict_proba(X_test_stage2)

    df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
    df_pred["Player"] = players_stage2
    df_pred["is_rookie"] = is_rookie_stage2

    results = {
        "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
        "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
        "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
        "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
        "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
    }

    with open("classification_result_2025.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)



Uzyskany score:
- Total score: 276/450
- first all-nba team: 68 points
- second all-nba team: 49 points
- third all-nba team: 47 points
- first rookie all-nba team: 56 points
- second rookie all-nba team: 56 points

## Stacking CLF (XGB, GBC, LR) - Training and Evaluation

- 1_STAGE: XGB
- 2_STAGE: Stacking CLF (XGB, GBC, LR)

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# === Flaga logowania
log_to_mlflow = True

if log_to_mlflow:
    import mlflow
    import mlflow.sklearn
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("NBA Final Prediction 2025")

# === 1. Wczytanie danych
df = pd.read_csv("databases/nba_dataset_2010_2025.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]
df["target_binary"] = (df["target"] > 0).astype(int)

train_mask = df["season"] < 2025
test_mask = df["season"] == 2025
X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)
y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: Binary classifier
model_bin = XGBClassifier(
    objective="binary:logistic",
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
stage1_preds = model_bin.predict(X_test)

# === 3. Dane do Stage 2
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)
X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
stage1_mask = pd.Series(stage1_preds == 1, index=players_test.index)
players_stage2 = players_test[stage1_mask].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_mask].reset_index(drop=True)

# === 4. StackingClassifier
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)
gbc = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
logreg_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=42
    )
)
stacking_model = StackingClassifier(
    estimators=[
        ("xgb", xgb),
        ("gbc", gbc)
    ],
    final_estimator=logreg_pipe,
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === 5. Trening + predykcja
if log_to_mlflow:
    with mlflow.start_run(run_name="Stacking_XGB_GBC_LogReg_2025"):
        # log stage 1 binary model
        mlflow.log_params({f"stage1__{k}": v for k, v in model_bin.get_params().items()})
        # log stacking base models
        mlflow.log_params({f"stage2_xgb__{k}": v for k, v in xgb.get_params().items()})
        mlflow.log_params({f"stage2_gbc__{k}": v for k, v in gbc.get_params().items()})
        mlflow.log_param("stage2_final_estimator", "StandardScaler + LogisticRegression")

        stacking_model.fit(X_train_stage2, y_train_stage2)
        probas_stage2 = stacking_model.predict_proba(X_test_stage2)

        # === 6. Piątki
        df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
        df_pred["Player"] = players_stage2
        df_pred["is_rookie"] = is_rookie_stage2
        df_pred["all_nba_score"] = df_pred[[1, 2, 3]].max(axis=1)

        top15_allnba = df_pred.sort_values("all_nba_score", ascending=False).head(15)
        results = {
            "first all-nba team": top15_allnba.iloc[:5]["Player"].tolist(),
            "second all-nba team": top15_allnba.iloc[5:10]["Player"].tolist(),
            "third all-nba team": top15_allnba.iloc[10:15]["Player"].tolist()
        }

        ordinal = ["first", "second"]
        already_rookies = set()
        for idx, class_id in enumerate([4, 5]):
            rookies = df_pred[(df_pred["is_rookie"] == 1) & (~df_pred["Player"].isin(already_rookies))]
            top5_rookies = rookies.sort_values(class_id, ascending=False).head(5)
            results[f"{ordinal[idx]} rookie all-nba team"] = top5_rookies["Player"].tolist()
            already_rookies.update(top5_rookies["Player"])

        with open("classification_result_2025.json", "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        mlflow.log_artifact("classification_result_2025.json")
        mlflow.set_tag("pipeline", "two_stage_stacking_classifier")
        mlflow.set_tag("prediction_season", "2025")

else:
    stacking_model.fit(X_train_stage2, y_train_stage2)
    probas_stage2 = stacking_model.predict_proba(X_test_stage2)

    df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
    df_pred["Player"] = players_stage2
    df_pred["is_rookie"] = is_rookie_stage2
    df_pred["all_nba_score"] = df_pred[[1, 2, 3]].max(axis=1)

    top15_allnba = df_pred.sort_values("all_nba_score", ascending=False).head(15)
    results = {
        "first all-nba team": top15_allnba.iloc[:5]["Player"].tolist(),
        "second all-nba team": top15_allnba.iloc[5:10]["Player"].tolist(),
        "third all-nba team": top15_allnba.iloc[10:15]["Player"].tolist()
    }

    ordinal = ["first", "second"]
    already_rookies = set()
    for idx, class_id in enumerate([4, 5]):
        rookies = df_pred[(df_pred["is_rookie"] == 1) & (~df_pred["Player"].isin(already_rookies))]
        top5_rookies = rookies.sort_values(class_id, ascending=False).head(5)
        results[f"{ordinal[idx]} rookie all-nba team"] = top5_rookies["Player"].tolist()
        already_rookies.update(top5_rookies["Player"])

    with open("classification_result_2025.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)




Uzyskany score:
- Total score: 276/450
- first all-nba team: 68 points
- second all-nba team: 49 points
- third all-nba team: 47 points
- first rookie all-nba team: 56 points
- second rookie all-nba team: 56 points

### Prediction Probabilities


In [ ]:
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2


In [ ]:
# Zapis pełnych prawdopodobieństw dla graczy z stage2
df_pred_sorted = df_pred.sort_values(by=1, ascending=False)  # możesz też sortować po max(axis=1)
df_pred_sorted.to_csv("stage2_all_players_probs.csv", index=False)


In [ ]:
len(X_test_stage2)

In [ ]:
# === Stage 1: prawdopodobieństwa (wszyscy gracze sezonu 2023)
stage1_probas = model_bin.predict_proba(X_test)[:, 1]  # prawdopodobieństwo klasy 1 (nagroda)

df_stage1_probs = pd.DataFrame({
    "Player": players_test,
    "is_rookie": is_rookie,
    "prob_has_award": stage1_probas
}).sort_values("prob_has_award", ascending=False)
df_stage1_probs.to_csv("stage1_probs.csv", index=False)

# === Stage 2: prawdopodobieństwa klas 1–5 (tylko stage1_preds == 1)
df_stage2_probs = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_stage2_probs["Player"] = players_stage2
df_stage2_probs["is_rookie"] = is_rookie_stage2
df_stage2_probs["max_prob"] = df_stage2_probs[[1, 2, 3, 4, 5]].max(axis=1)
df_stage2_probs = df_stage2_probs.sort_values("max_prob", ascending=False)
df_stage2_probs.drop(columns=["max_prob"], inplace=True)
df_stage2_probs.to_csv("stage2_probs.csv", index=False)

## Stacking CLF (XGB, GBC, LR) Params Tuning 

- 1_STAGE: XGB
- 2_STAGE: Stacking CLF (XGB, GBC, LR)

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV

# === Flaga logowania ===
log_to_mlflow = False

if log_to_mlflow:
    import mlflow
    import mlflow.sklearn
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("Ensemble CLF Hyperparam Tuning")

# === Dane
df = pd.read_csv("databases/nba_dataset_2000_2025.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]
df["target_binary"] = (df["target"] > 0).astype(int)

train_mask = df["season"] < 2025
X_train = df[train_mask].drop(columns=drop_cols)
y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

# === Stage 1: Binary classifier
model_bin = XGBClassifier(objective="binary:logistic", use_label_encoder=False, eval_metric="logloss", random_state=42)
sample_weight = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight)

# === Dane do Stage 2
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)

# === Stacking model
xgb = XGBClassifier(objective="multi:softprob", num_class=5, use_label_encoder=False, eval_metric="mlogloss", random_state=42)
gbc = GradientBoostingClassifier(random_state=42)
logreg_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", multi_class="multinomial", solver="lbfgs", random_state=42))

stack = StackingClassifier(
    estimators=[("xgb", xgb), ("gbc", gbc)],
    final_estimator=logreg_pipe,
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === Parametry do przeszukania
param_dist = {
    # XGBoost
    "xgb__n_estimators": [100, 150, 180, 200, 220, 250, 300],
    "xgb__max_depth": [3, 4, 5, 6, 7],
    "xgb__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1, 0.12],
    "xgb__subsample": [0.6, 0.7, 0.8, 1.0],
    "xgb__colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "xgb__gamma": [0, 0.5, 1, 2],

    # GradientBoostingClassifier
    "gbc__n_estimators": [100, 150, 180, 200, 220, 250, 300],
    "gbc__max_depth": [3, 4, 5, 6],
    "gbc__learning_rate": [0.01, 0.05, 0.08, 0.1, 0.15, 0.18, 0.2],
    "gbc__subsample": [0.6, 0.8, 1.0],
    "gbc__min_samples_split": [2, 5, 10],
    "gbc__min_samples_leaf": [1, 3, 5],

    # Logistic Regression (final estimator)
    "final_estimator__logisticregression__C": [0.01, 0.1, 0.5, 1, 5, 10, 20],
    "final_estimator__logisticregression__penalty": ["l2"],  # l1 only works with liblinear
    "final_estimator__logisticregression__solver": ["lbfgs", "saga"]
}

search = RandomizedSearchCV(
    estimator=stack,
    param_distributions=param_dist,
    n_iter=30,  # większa liczba kombinacji
    cv=3,
    scoring="f1_macro",
    verbose=2,
    n_jobs=-1,
    random_state=42
)


# === Fit i logowanie
if log_to_mlflow:
    with mlflow.start_run(run_name="Deep_Tuning_2010-2024"):
        search.fit(X_train_stage2, y_train_stage2)
        best_model = search.best_estimator_
        mlflow.log_params(search.best_params_)
        mlflow.log_metric("best_f1_macro", search.best_score_)
        mlflow.set_tag("model", "StackingClassifier")
        mlflow.set_tag("stage", "param_search")
else:
    search.fit(X_train_stage2, y_train_stage2)
    best_model = search.best_estimator_


In [8]:
search.best_params_

{'xgb__subsample': 1.0,
 'xgb__n_estimators': 300,
 'xgb__max_depth': 4,
 'xgb__learning_rate': 0.01,
 'xgb__gamma': 1,
 'xgb__colsample_bytree': 0.6,
 'gbc__subsample': 1.0,
 'gbc__n_estimators': 150,
 'gbc__min_samples_split': 10,
 'gbc__min_samples_leaf': 1,
 'gbc__max_depth': 4,
 'gbc__learning_rate': 0.2,
 'final_estimator__logisticregression__solver': 'lbfgs',
 'final_estimator__logisticregression__penalty': 'l2',
 'final_estimator__logisticregression__C': 1}

In [ ]:
# Zapis pełnych prawdopodobieństw dla graczy z stage2
df_pred_sorted = df_pred.sort_values(by=1, ascending=False)  # możesz też sortować po max(axis=1)
df_pred_sorted.to_csv("stage2_all_players_probs.csv", index=False)

 ## Stacking CLF (XGB, GBC, LR) with Hyperparameters assigned

- 1_STAGE: XGB
- 2_STAGE: Stacking CLF (XGB, GBC, LR)

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# === Flaga logowania do MLflow
log_to_mlflow = False

if log_to_mlflow:
    import mlflow
    import mlflow.sklearn
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("NBA Final Prediction 2025")

# === Bazowe modele z best_params
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    eval_metric="mlogloss",
    n_estimators=200,
    max_depth=6,
    gamma=1,
    colsample_bytree=0.7,
    learning_rate=0.1,
    random_state=42
)

gbc = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.01,
    subsample=1.0,
    random_state=42
)

logreg_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        C=0.1,
        random_state=42
    )
)

# === 1. Dane
df = pd.read_csv("databases/nba_dataset_2010_2025.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]
df["target_binary"] = (df["target"] > 0).astype(int)

train_mask = df["season"] < 2025
test_mask = df["season"] == 2025

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)
y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1
model_bin = XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=42)
sample_weight = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight)
stage1_preds = model_bin.predict(X_test)

# === 3. Stage 2 dane
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)
X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
players_stage2 = players_test[stage1_preds == 1].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_preds == 1].reset_index(drop=True)

# === 4. Finalny stacking model
stacking_best = StackingClassifier(
    estimators=[("xgb", xgb), ("gbc", gbc)],
    final_estimator=logreg_pipe,
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === 5. Trening + logowanie
if log_to_mlflow:
    with mlflow.start_run(run_name="V3_Stacking_XGB_GBC_LogReg_2025"):
        stacking_best.fit(X_train_stage2, y_train_stage2)

        # loguj parametry modeli bazowych
        mlflow.log_params({f"xgb__{k}": v for k, v in xgb.get_params().items()})
        mlflow.log_params({f"gbc__{k}": v for k, v in gbc.get_params().items()})
        mlflow.log_param("final_estimator", "StandardScaler + LogisticRegression(C=10)")

        mlflow.set_tag("pipeline", "stacking_classifier")
        mlflow.set_tag("prediction_season", "2025")
else:
    stacking_best.fit(X_train_stage2, y_train_stage2)

# === 6. Predykcja
probas_stage2 = stacking_best.predict_proba(X_test_stage2)

# === 7. Zbudowanie piątek
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2
df_pred["all_nba_score"] = df_pred[[1, 2, 3]].sum(axis=1)

top15 = df_pred.sort_values("all_nba_score", ascending=False).head(15).copy()
print("Top 15 graczy sortowanych po all_nba_score:")
print(top15[["Player", 1, "all_nba_score"]])

ranks = {
    "first all-nba team": top15.sort_values(1, ascending=False)["Player"].tolist(),
    "second all-nba team": top15.sort_values(2, ascending=False)["Player"].tolist(),
    "third all-nba team": top15.sort_values(3, ascending=False)["Player"].tolist()
}

results = {team: [] for team in ranks}
used_players = set()

while any(len(results[team]) < 5 for team in results):
    for team in ["first all-nba team", "second all-nba team", "third all-nba team"]:
        for player in ranks[team]:
            if player not in used_players:
                results[team].append(player)
                used_players.add(player)
                break

# === 8. Rookie teams
rookies_df = df_pred[df_pred["is_rookie"] == 1].copy()
rookie_ranks = {
    "first rookie all-nba team": rookies_df.sort_values(4, ascending=False)["Player"].tolist(),
    "second rookie all-nba team": rookies_df.sort_values(5, ascending=False)["Player"].tolist()
}

results.update({team: [] for team in rookie_ranks})
used_rookies = set()


while any(len(results[team]) < 5 for team in rookie_ranks):
    for team in ["first rookie all-nba team", "second rookie all-nba team"]:
        for player in rookie_ranks[team]:
            if player not in used_rookies:
                results[team].append(player)
                used_rookies.add(player)
                break

# === 9. Zapis JSON + log artefaktu
output_path = "classification_result_2025.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

if log_to_mlflow:
    mlflow.log_artifact(output_path)




### Saving Prediction Probability - Players stage 2

In [ ]:
# Zapis pełnych prawdopodobieństw dla graczy z stage2
df_pred_sorted = df_pred.sort_values(by=1, ascending=False)  # możesz też sortować po max(axis=1)
df_pred_sorted.to_csv("stage2_all_players_probs.csv", index=False)

In [10]:
df_pred_sorted1 = df_pred.sort_values(by=1, ascending=False)
df_pred_sorted2 = df_pred.sort_values(by=2, ascending=False)
df_pred_sorted3 = df_pred.sort_values(by=3, ascending=False)

In [11]:
df_pred_sorted1

,1,2,3,4,5,Player,is_rookie,all_nba_score
18,0.847015,0.138362,0.014123,0.000165,0.000335,Shai Gilgeous-Alexander,0,0.999500
7,0.841800,0.142636,0.015016,0.000196,0.000352,Giannis Antetokounmpo,0,0.999452
17,0.819059,0.160980,0.019375,0.000189,0.000397,Nikola Jokić,0,0.999414
12,0.684854,0.252715,0.061563,0.000355,0.000513,Jayson Tatum,0,0.999132
1,0.341949,0.398707,0.257718,0.001098,0.000528,Anthony Edwards,0,0.998374
10,0.338575,0.384942,0.273712,0.002140,0.000631,James Harden,0,0.997229
19,0.223660,0.372452,0.402252,0.001293,0.000344,Stephen Curry,0,0.998364
13,0.166578,0.370506,0.461995,0.000523,0.000398,Karl-Anthony Towns,0,0.999079
8,0.103786,0.327553,0.566195,0.002106,0.000359,Jalen Brunson,0,0.997535
15,0.081115,0.331009,0.577617,0.009700,0.000558,LeBron James,0,0.989741


In [12]:
df_pred_sorted2

,1,2,3,4,5,Player,is_rookie,all_nba_score
1,0.341949,0.398707,0.257718,0.001098,0.000528,Anthony Edwards,0,0.998374
10,0.338575,0.384942,0.273712,0.002140,0.000631,James Harden,0,0.997229
19,0.223660,0.372452,0.402252,0.001293,0.000344,Stephen Curry,0,0.998364
13,0.166578,0.370506,0.461995,0.000523,0.000398,Karl-Anthony Towns,0,0.999079
15,0.081115,0.331009,0.577617,0.009700,0.000558,LeBron James,0,0.989741
8,0.103786,0.327553,0.566195,0.002106,0.000359,Jalen Brunson,0,0.997535
6,0.067725,0.299373,0.600020,0.032333,0.000549,Evan Mobley,0,0.967117
5,0.055227,0.285237,0.651073,0.008012,0.000451,Donovan Mitchell,0,0.991537
21,0.062053,0.264867,0.638633,0.034153,0.000294,Tyrese Haliburton,0,0.965553
3,0.053899,0.257182,0.670760,0.017836,0.000323,Cade Cunningham,0,0.981841


In [13]:
df_pred_sorted3

,1,2,3,4,5,Player,is_rookie,all_nba_score
21,0.054390,0.293101,0.634994,0.010043,0.007472,Tyrese Haliburton,0,0.982485
9,0.055930,0.297948,0.629189,0.009495,0.007438,Jalen Williams,0,0.983067
10,0.059487,0.309674,0.614265,0.009736,0.006839,James Harden,0,0.983425
3,0.065253,0.315915,0.600313,0.010633,0.007886,Cade Cunningham,0,0.981480
15,0.064578,0.325225,0.592955,0.009756,0.007487,LeBron James,0,0.982757
6,0.077663,0.320887,0.579055,0.013282,0.009113,Evan Mobley,0,0.977605
19,0.067522,0.340977,0.577124,0.008623,0.005754,Stephen Curry,0,0.985623
5,0.084223,0.339129,0.556377,0.011341,0.008930,Donovan Mitchell,0,0.979729
8,0.107886,0.388963,0.486841,0.010162,0.006148,Jalen Brunson,0,0.983690
13,0.157432,0.442271,0.383439,0.012635,0.004223,Karl-Anthony Towns,0,0.983142


In [ ]:
len( X_train[y_train_bin == 1])

### Feature Importance XGBoost of Stacking CLF

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

# Sięgamy po estimator XGBoost z warstwy bazowej
xgb_model = stacking_best.named_estimators_["xgb"]

# Rysujemy importance (gain / weight / cover)
plot_importance(xgb_model, max_num_features=20, importance_type="gain")
plt.title("XGBoost Feature Importance (warstwa bazowa)")
plt.tight_layout()
plt.show()


### Permutation Importance of Stacking CLF

In [ ]:
# Zakładamy, że target znajduje się w kolumnie o nazwie 'target'
y_test_target = df.loc[test_mask, "target"]

# Filtrujemy po tych samych predykcjach co stage1
y_test_stage2 = y_test_target[stage1_preds == 1].map(class_map).reset_index(drop=True)



from sklearn.inspection import permutation_importance
import pandas as pd
import matplotlib.pyplot as plt

# === 1. Załóżmy, że masz:
# stacking_best_stage2 — wytrenowany model dla stage 2 (np. StackingClassifier)
# X_test_stage2 — dane testowe z klasyfikacji binarnej stage 1
# y_test_stage2 — mapowane etykiety klas (0–4)

# === 2. Obliczamy permutation importance
result = permutation_importance(
    estimator=stacking_best,
    X=X_test_stage2,
    y=y_test_stage2,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# === 3. Formatowanie wyników
importance_df = pd.DataFrame({
    "feature": X_test_stage2.columns,
    "importance": result.importances_mean,
    "std": result.importances_std
}).sort_values(by="importance", ascending=False)

# === 4. Wyświetlenie top 15 cech
print("Top 15 cech wg permutation importance (Stage 2):")
print(importance_df.head(15))

# === 5. (Opcjonalnie) Wykres
plt.figure(figsize=(10, 6))
plt.barh(importance_df.head(15)["feature"], importance_df.head(15)["importance"], xerr=importance_df.head(15)["std"])
plt.xlabel("Permutation Importance")
plt.title("Najważniejsze cechy w Stacking Stage 2")
plt.gca().invert_yaxis()
plt.grid(True)
plt.tight_layout()
plt.show()


## Stacking CLF (XGB, GBC, LR) Tuning with more Params and Prediction

- 1_STAGE: XGB
- 2_STAGE: Stacking CLF (XGB, GBC, LR)

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV

# === Flaga logowania
log_to_mlflow = True

if log_to_mlflow:
    import mlflow
    import mlflow.sklearn
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("Ensemble CLF Hyperparam Tuning")

# === 1. Wczytanie danych
df = pd.read_csv("databases/nba_dataset_2010_2025.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]
df["target_binary"] = (df["target"] > 0).astype(int)

train_mask = df["season"] < 2025
test_mask = df["season"] == 2025

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)
y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]
players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: klasyfikacja binarna
model_bin = XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=42)
sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
stage1_preds = model_bin.predict(X_test)

# === 3. Dane Stage 2
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}

X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)
X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
players_stage2 = players_test[stage1_preds == 1].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_preds == 1].reset_index(drop=True)

# === 4. Składniki stacking
model_xgb = XGBClassifier(objective="multi:softprob", num_class=5, eval_metric="mlogloss", random_state=42)
model_gbc = GradientBoostingClassifier(random_state=42)
logreg_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=42
    )
)

stacking_model = StackingClassifier(
    estimators=[
        ("xgb", model_xgb),
        ("gbc", model_gbc)
    ],
    final_estimator=logreg_pipeline,
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === 5. Rozszerzona przestrzeń hiperparametrów
param_distributions = {
    "final_estimator__logisticregression__C": [0.001, 0.01, 0.1, 1, 5, 10],
    "gbc__n_estimators": [100, 200, 300, 500],
    "gbc__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gbc__max_depth": [3, 4, 5, 6],
    "gbc__subsample": [0.7, 0.8, 1.0],
    "xgb__n_estimators": [200, 300, 500],
    "xgb__learning_rate": [0.05, 0.1, 0.2],
    "xgb__max_depth": [3, 4, 5, 6],
    "xgb__colsample_bytree": [0.7, 0.9, 1.0],
    "xgb__gamma": [0, 1, 5]
}

search = RandomizedSearchCV(
    estimator=stacking_model,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# === 6. Fit + logowanie MLflow
if log_to_mlflow:
    with mlflow.start_run(run_name="Stacking_DeepSearch_2025"):
        search.fit(X_train_stage2, y_train_stage2)
        best_model = search.best_estimator_

        mlflow.log_params(search.best_params_)
        mlflow.log_metric("best_f1_macro", search.best_score_)
        mlflow.set_tag("pipeline", "stacking_classifier_tuned")
        mlflow.set_tag("stage", "hyperparam_tuning")
        mlflow.set_tag("prediction_season", "2025")
else:
    search.fit(X_train_stage2, y_train_stage2)
    best_model = search.best_estimator_

# === 7. Predykcja
probas_stage2 = best_model.predict_proba(X_test_stage2)

df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

# === 8. Zapis JSON
with open("classification_result_2025.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

if log_to_mlflow:
    mlflow.log_artifact("classification_result_2025.json")

print("Gotowe: log =", "włączony" if log_to_mlflow else "wyłączony")


## Stacking CLF (XGB + MLP + NB) Params Tuning with Prediction

- 1_STAGE: XGB
- 2_STAGE: Stacking CLF (XGB, MLP, NB)

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight

# === Flaga
log_to_mlflow = False

if log_to_mlflow:
    import mlflow
    import mlflow.sklearn
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("Ensemble CLF Hyperparam Tuning")

# === Dane
df = pd.read_csv("databases/nba_dataset_2000_2025.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]
df["target_binary"] = (df["target"] > 0).astype(int)

X_train = df[df["season"] < 2025].drop(columns=drop_cols)
X_test = df[df["season"] == 2025].drop(columns=drop_cols)
y_train_bin = df[df["season"] < 2025]["target_binary"]
y_train_full = df[df["season"] < 2025]["target"]

players_test = df[df["season"] == 2025]["Player"].reset_index(drop=True)
is_rookie = df[df["season"] == 2025]["is_rookie"].reset_index(drop=True)

# === Stage 1
model_bin = XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=42)
sample_weight = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight)
stage1_preds = model_bin.predict(X_test)

# === Stage 2 dane
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)

X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
players_stage2 = players_test[stage1_preds == 1].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_preds == 1].reset_index(drop=True)

# === Stacking
stacking = StackingClassifier(
    estimators=[
        ("xgb", XGBClassifier(objective="multi:softprob", num_class=5, eval_metric="mlogloss", random_state=42)),
        ("mlp", MLPClassifier(max_iter=1000, random_state=42)),
        ("nb", GaussianNB())
    ],
    final_estimator=make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            multi_class="multinomial",
            solver="lbfgs",
            random_state=42
        )
    ),
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === Rozszerzona przestrzeń hiperparametrów
param_distributions = {
    # === MLPClassifier
    "mlp__hidden_layer_sizes": [(64,), (100,), (128,), (64, 64), (100, 50)],
    "mlp__alpha": [0.0001, 0.001, 0.01, 0.1],
    "mlp__learning_rate_init": [0.001, 0.01, 0.1],
    "mlp__activation": ["relu", "tanh"],

    # === XGBoost
    "xgb__n_estimators": [100, 200, 300, 400],
    "xgb__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "xgb__max_depth": [3, 4, 5, 6],
    "xgb__colsample_bytree": [0.6, 0.8, 1.0],
    "xgb__subsample": [0.7, 0.9, 1.0],
    "xgb__gamma": [0, 1, 5],

    # === Logistic Regression (meta-klasyfikator)
    "final_estimator__logisticregression__C": [0.1, 1, 5, 10, 20]
}

# === RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=stacking,
    param_distributions=param_distributions,
    n_iter=75,  # zwiększona liczba iteracji
    scoring="f1_macro",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# === Fit & logowanie
if log_to_mlflow:
    with mlflow.start_run(run_name="XGB+MLP+NB_Stacking_Tuning_2025"):
        search.fit(X_train_stage2, y_train_stage2)
        best_model = search.best_estimator_
        mlflow.log_params(search.best_params_)
        mlflow.log_metric("best_f1_macro", search.best_score_)
        mlflow.set_tag("model", "stacking_xgb_mlp_nb")
        mlflow.set_tag("stage", "search + prediction")
        mlflow.set_tag("season", "2025")
else:
    search.fit(X_train_stage2, y_train_stage2)
    best_model = search.best_estimator_

# === Predykcja
probas = best_model.predict_proba(X_test_stage2)
df_pred = pd.DataFrame(probas, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

with open("classification_result_2025.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

if log_to_mlflow:
    mlflow.log_artifact("classification_result_2025.json")

print("Gotowe: log =", "włączony" if log_to_mlflow else "wyłączony")


## Stacking CLF (XGB + GBC + NB) Params Tuning and Prediction

- 1_STAGE: XGB
- 2_STAGE: Stacking CLF (XGB, GBC, NB)

In [ ]:
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight
import pandas as pd
import json
import mlflow
import mlflow.sklearn

# === Flaga
log_to_mlflow = True

if log_to_mlflow:
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("Ensemble CLF Hyperparam Tuning")

# === Dane
df = pd.read_csv("databases/nba_dataset_2010_2025.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]
df["target_binary"] = (df["target"] > 0).astype(int)

X_train = df[df["season"] < 2025].drop(columns=drop_cols)
X_test = df[df["season"] == 2025].drop(columns=drop_cols)
y_train_bin = df[df["season"] < 2025]["target_binary"]
y_train_full = df[df["season"] < 2025]["target"]

players_test = df[df["season"] == 2025]["Player"].reset_index(drop=True)
is_rookie = df[df["season"] == 2025]["is_rookie"].reset_index(drop=True)

# === Stage 1
model_bin = XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=42)
sample_weight = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight)
stage1_preds = model_bin.predict(X_test)

# === Stage 2 dane
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)
X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
players_stage2 = players_test[stage1_preds == 1].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_preds == 1].reset_index(drop=True)

# === Stacking setup
stacking = StackingClassifier(
    estimators=[
        ("xgb", XGBClassifier(objective="multi:softprob", num_class=5, eval_metric="mlogloss", random_state=42)),
        ("gbc", GradientBoostingClassifier(random_state=42)),
        ("nb", GaussianNB())
    ],
    final_estimator=make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            multi_class="multinomial",
            solver="lbfgs",
            random_state=42
        )
    ),
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === Rozszerzone parametry
param_distributions = {
    "gbc__n_estimators": [100, 200, 300, 400],
    "gbc__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gbc__max_depth": [3, 4, 5, 6],
    "gbc__subsample": [0.7, 0.8, 1.0],
    "xgb__n_estimators": [200, 300, 400],
    "xgb__learning_rate": [0.05, 0.1, 0.2],
    "xgb__max_depth": [3, 4, 5],
    "xgb__colsample_bytree": [0.7, 0.9, 1.0],
    "xgb__gamma": [0, 1, 5],
    "final_estimator__logisticregression__C": [0.1, 1, 5, 10, 20]
}

search = RandomizedSearchCV(
    estimator=stacking,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# === Fit & MLflow
if log_to_mlflow:
    with mlflow.start_run(run_name="Stacking_XGB+GBC+NB_Tuning_2025"):
        search.fit(X_train_stage2, y_train_stage2)
        best_model = search.best_estimator_
        mlflow.log_params(search.best_params_)
        mlflow.log_metric("best_f1_macro", search.best_score_)
        mlflow.set_tag("model", "stacking_xgb_gbc_nb")
        mlflow.set_tag("stage", "search + prediction")
        mlflow.set_tag("season", "2025")
        mlflow.sklearn.log_model(best_model, "stacking_model")
else:
    search.fit(X_train_stage2, y_train_stage2)
    best_model = search.best_estimator_

# === Predykcja
probas = best_model.predict_proba(X_test_stage2)
df_pred = pd.DataFrame(probas, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

with open("classification_result_2025.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

if log_to_mlflow:
    mlflow.log_artifact("classification_result_2025.json")



- Total score: 286/450
- first all-nba team: 68 points
- second all-nba team: 42 points
- third all-nba team: 40 points
- first rookie all-nba team: 68 points
- second rookie all-nba team: 68 points

## Voting CLF with 3 Stacked Classifiers, Prediction

- 1_STAGE: XGB
- 2_STAGE: Voting CLF (3 Stacked CLF) - (XGB, GBC, LR); (XGB, GBC, MLP); (XGB, GBC, XGB)

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# === 1. Wczytanie danych ===
df = pd.read_csv("databases/nba_dataset_2010_2025.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]

df["target_binary"] = (df["target"] > 0).astype(int)

train_mask = df["season"] < 2025
test_mask = df["season"] == 2025

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)

y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: klasyfikacja binarna
model_bin = XGBClassifier(
    objective="binary:logistic",
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
stage1_preds = model_bin.predict(X_test)

# === 3. Przygotowanie danych dla stage 2
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)

X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
stage1_mask = pd.Series(stage1_preds == 1, index=players_test.index)

players_stage2 = players_test[stage1_mask].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_mask].reset_index(drop=True)

# === 4. Wspólne modele bazowe
base_estimators = [
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=5,
        use_label_encoder=False,
        eval_metric="mlogloss",
        random_state=42
    )),
    ("gbc", GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
]

# === 5. Trzy stackingi

# LogisticRegression (ze scalerem)
stack_logreg = StackingClassifier(
    estimators=base_estimators,
    final_estimator=make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            multi_class="multinomial",
            solver="lbfgs",
            random_state=42
        )
    ),
    stack_method="predict_proba",
    cv=3,
    n_jobs=-1
)

# MLPClassifier
stack_mlp = StackingClassifier(
    estimators=base_estimators,
    final_estimator=make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
    ),
    stack_method="predict_proba",
    cv=3,
    n_jobs=-1
)

# XGBClassifier jako meta-model
stack_xgb = StackingClassifier(
    estimators=base_estimators,
    final_estimator=XGBClassifier(
        objective="multi:softprob",
        num_class=5,
        use_label_encoder=False,
        eval_metric="mlogloss",
        random_state=42
    ),
    stack_method="predict_proba",
    cv=3,
    n_jobs=-1
)

# === 6. VotingClassifier z 3 stackingów
voting_on_stackings = VotingClassifier(
    estimators=[
        ("stack_logreg", stack_logreg),
        ("stack_mlp", stack_mlp),
        ("stack_xgb", stack_xgb)
    ],
    voting="soft"
)

# === 7. Trening + predykcja
voting_on_stackings.fit(X_train_stage2, y_train_stage2)
probas_stage2 = voting_on_stackings.predict_proba(X_test_stage2)

# === 8. Tworzenie piątek
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

# === 9. Zapis JSON
with open("classification_result_2025.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)



- Total score: 274/450
- first all-nba team: 66 points
- second all-nba team: 49 points
- third all-nba team: 47 points
- first rookie all-nba team: 56 points
- second rookie all-nba team: 56 points

## Voting CLF with Stacking Classifier, XGB solo - Params Tuning, Prediction

- 1_STAGE: XGB
- 2_STAGE: Voting CLF: Stacked CLF(XGB, GBC, LR); XGB

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV

# === 1. Wczytanie danych ===
df = pd.read_csv("databases/nba_dataset_2010_2025.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]

df["target_binary"] = (df["target"] > 0).astype(int)
train_mask = df["season"] < 2025
test_mask = df["season"] == 2025

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)

y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: klasyfikacja binarna
model_bin = XGBClassifier(
    objective="binary:logistic",
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
stage1_preds = model_bin.predict(X_test)

# === 3. Stage 2 dane
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)

X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
stage1_mask = pd.Series(stage1_preds == 1, index=players_test.index)

players_stage2 = players_test[stage1_mask].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_mask].reset_index(drop=True)

# === 4. Składniki stackingu
model_xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)

model_gbc = GradientBoostingClassifier(random_state=42)

logreg_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=42
    )
)

stacking_model = StackingClassifier(
    estimators=[
        ("xgb", model_xgb),
        ("gbc", model_gbc)
    ],
    final_estimator=logreg_pipeline,
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === 5. RandomizedSearchCV na stackingu
param_distributions = {
    "final_estimator__logisticregression__C": [0.001, 0.01, 0.1, 1, 5, 10],
    "gbc__n_estimators": [100, 200, 300, 500],
    "gbc__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gbc__max_depth": [3, 4, 5, 6],
    "gbc__subsample": [0.7, 0.8, 1.0],
    "xgb__n_estimators": [200, 300, 500],
    "xgb__learning_rate": [0.05, 0.1, 0.2],
    "xgb__max_depth": [3, 4, 5, 6],
    "xgb__colsample_bytree": [0.7, 0.9, 1.0],
    "xgb__gamma": [0, 1, 5]
}

search = RandomizedSearchCV(
    estimator=stacking_model,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train_stage2, y_train_stage2)
best_stack = search.best_estimator_

# === 6. Niezależny model: XGBClassifier
model_xgb_solo = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    use_label_encoder=False,
    eval_metric="mlogloss",
    n_estimators=300,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)
model_xgb_solo.fit(X_train_stage2, y_train_stage2)

# === 7. VotingClassifier (stacking + XGB)
voting_combo = VotingClassifier(
    estimators=[
        ("stacking", best_stack),
        ("xgb_solo", model_xgb_solo)
    ],
    voting="soft",
    weights=[2, 1]  # większe zaufanie do stackingu
)

voting_combo.fit(X_train_stage2, y_train_stage2)
probas_stage2 = voting_combo.predict_proba(X_test_stage2)

# === 8. Tworzenie piątek
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

# === 9. Zapis JSON
with open("classification_result_2025.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)



Uzyskany score:
- Total score: 326/450
- first all-nba team: 68 points
- second all-nba team: 68 points
- third all-nba team: 54 points
- first rookie all-nba team: 68 points
- second rookie all-nba team: 68 points

In [ ]:
best_stack.get_params()

In [ ]:
model_xgb_solo.get_params()

## Stage 1: MLP; Stage 2: Voiting CLF (stack, rf, et); Prediction

- 1_STAGE: MLP
- 2_STAGE: Voting CLF: (MLP, XBG, LR); RF; ET

In [ ]:
import pandas as pd
import json
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier, VotingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

# === 1. Wczytanie danych ===
df = pd.read_csv("databases/nba_dataset_2010_2025.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]

df["target_binary"] = (df["target"] > 0).astype(int)
train_mask = df["season"] < 2025
test_mask = df["season"] == 2025

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)

y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: klasyfikacja binarna (XGBoost) ===
model_bin = XGBClassifier(
    objective="binary:logistic",
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
stage1_preds = model_bin.predict(X_test)

# === 3. Stage 2 dane ===
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)

X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
stage1_mask = pd.Series(stage1_preds == 1, index=players_test.index)

players_stage2 = players_test[stage1_mask].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_mask].reset_index(drop=True)

# === 4. Stacking model ===
model_mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
model_xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)

stacking_model = StackingClassifier(
    estimators=[
        ("mlp", model_mlp),
        ("xgb", model_xgb),
        ("lr", logreg_pipeline)
    ],
    final_estimator=make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            multi_class="multinomial",
            solver="lbfgs",
            random_state=42
        )
    ),
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === 5. VotingClassifier (stack + RF + ET) ===
voting_combo = VotingClassifier(
    estimators=[
        ("stacking", stacking_model),
        ("rf", RandomForestClassifier(n_estimators=200, random_state=42)),
        ("et", ExtraTreesClassifier(n_estimators=200, random_state=42))
    ],
    voting="soft",
    weights=[3, 1, 1]
)

voting_combo.fit(X_train_stage2, y_train_stage2)
probas_stage2 = voting_combo.predict_proba(X_test_stage2)

# === 6. Tworzenie piątek ===
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

# === 7. Zapis JSON ===
with open("classification_result_2025.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)


## Players divided in results teams - 1 Method

In [ ]:
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

results = {}
ordinal = ["first", "second", "third"]

# All-NBA teams (1st, 2nd, 3rd)
already_selected = set()
for idx, class_id in enumerate([1, 2, 3]):
    available = df_pred[~df_pred["Player"].isin(already_selected)]
    top5 = available.sort_values(class_id, ascending=False).head(5)
    team_name = f"{ordinal[idx]} all-nba team"
    team_players = top5["Player"].tolist()
    results[team_name] = team_players
    already_selected.update(team_players)

# All-Rookie teams (1st, 2nd)
already_rookies = set()
for idx, class_id in enumerate([4, 5]):
    available_rookies = df_pred[(df_pred["is_rookie"] == 1) & (~df_pred["Player"].isin(already_rookies))]
    top5 = available_rookies.sort_values(class_id, ascending=False).head(5)
    team_name = f"{ordinal[idx]} rookie all-nba team"
    team_players = top5["Player"].tolist()
    results[team_name] = team_players
    already_rookies.update(team_players)


# === 7. Zapis do JSON
with open("classification_result_2025.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

## Players divided in results teams - 2 Method

In [ ]:
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2
df_pred["all_nba_score"] = df_pred[[1, 2, 3]].sum(axis=1)

top15 = df_pred.sort_values("all_nba_score", ascending=False).head(15).copy()
#print("Top 15 graczy sortowanych po all_nba_score:")
#print(top15[["Player", 1, "all_nba_score"]])

ranks = {
    "first all-nba team": top15.sort_values(1, ascending=False)["Player"].tolist(),
    "second all-nba team": top15.sort_values(2, ascending=False)["Player"].tolist(),
    "third all-nba team": top15.sort_values(3, ascending=False)["Player"].tolist()
}

results = {team: [] for team in ranks}
used_players = set()

while any(len(results[team]) < 5 for team in results):
    for team in ["first all-nba team", "second all-nba team", "third all-nba team"]:
        for player in ranks[team]:
            if player not in used_players:
                results[team].append(player)
                used_players.add(player)
                break

# === 8. Rookie teams
rookies_df = df_pred[df_pred["is_rookie"] == 1].copy()
rookie_ranks = {
    "first rookie all-nba team": rookies_df.sort_values(4, ascending=False)["Player"].tolist(),
    "second rookie all-nba team": rookies_df.sort_values(5, ascending=False)["Player"].tolist()
}

results.update({team: [] for team in rookie_ranks})
used_rookies = set()


while any(len(results[team]) < 5 for team in rookie_ranks):
    for team in ["first rookie all-nba team", "second rookie all-nba team"]:
        for player in rookie_ranks[team]:
            if player not in used_rookies:
                results[team].append(player)
                used_rookies.add(player)
                break

# === 9. Zapis JSON + log artefaktu
output_path = "classification_result_2025.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)